In [43]:
# Member Check-In Behavior Analysis

## Objective
#Analyze member check-in activity to identify peak hours, usage patterns, and behavioral trends that can inform operational decisions.

## Data Source
#- PeopleVine Check-In Data

## Key Questions
#- What are the busiest hours of the day?
#- Who are the most and least engaged members?
#- What are the busiest days/months?
#- What trends exist in member engagement?

In [44]:
## Data Loading
#Importing and previewing the check-in dataset.

import pandas as pd

df = pd.read_excel('/Users/marshallmartin/Downloads/Data Analytics Finals/Check In Data.xlsx')

df.head()

,scan_date,first_name,last_name,membership_title,scan_location
0,2025-11-01 06:58:55,John,Iacoi,Member,Peoplevine Check-In
1,2025-11-01 07:39:30,Jerry,Noonan,Member,Peoplevine Check-In
2,2025-11-01 07:40:50,Barry,Greene,Member,Peoplevine Check-In
3,2025-11-01 07:40:53,Barry,Greene,Member,Peoplevine Check-In
4,2025-11-01 07:44:33,Mynor,Perez,Member,Peoplevine Check-In


In [5]:
## Data Preparation
#Cleaning and formatting date/time fields for analysis.

df['scan_date'] = pd.to_datetime(df['scan_date'], errors='coerce')

In [6]:
df.head()

,scan_date,first_name,last_name,membership_title,scan_location
0,2025-11-01 06:58:55,John,Iacoi,Member,Peoplevine Check-In
1,2025-11-01 07:39:30,Jerry,Noonan,Member,Peoplevine Check-In
2,2025-11-01 07:40:50,Barry,Greene,Member,Peoplevine Check-In
3,2025-11-01 07:40:53,Barry,Greene,Member,Peoplevine Check-In
4,2025-11-01 07:44:33,Mynor,Perez,Member,Peoplevine Check-In


In [7]:
#Identifying and removing duplicates across dataset

df.duplicated().sum()

np.int64(4663)

In [8]:
df[df.duplicated()]

,scan_date,first_name,last_name,membership_title,scan_location
44,2025-11-01 10:21:39,Susan,McAuliff,Member,NaN
67,2025-11-01 10:36:07,Kelly,Warner,Member,NaN
70,2025-11-01 10:36:31,Emily,Bosak,Member,NaN
121,2025-11-01 11:44:47,Meredith,Tedford,Member,NaN
122,2025-11-01 11:44:47,Meredith,Tedford,Member,NaN
...,...,...,...,...,...
53499,2026-01-30 19:48:44,Tonida,Vaka,Member,Peoplevine Check-In
53520,2026-01-30 20:05:56,Mark,Minassian,Member,Peoplevine Check-In
53533,2026-01-30 20:11:06,Christina,Bonfiglio,Member,Peoplevine Check-In
53536,2026-01-30 20:13:16,Camille,Martin,Member,Peoplevine Check-In


In [9]:
df = df.drop_duplicates(subset=['scan_date', 'first_name', 'last_name'])

In [10]:
df.duplicated().sum()

np.int64(0)

In [12]:
df['member_name'] = (
    df['first_name'].fillna('').astype(str).str.strip()
    + " " +
    df['last_name'].fillna('').astype(str).str.strip()
).str.strip()

In [45]:
### Highly Engaged Members
#This analysis identifies members with the highest frequency of visits.

#**Insight:**  
#A small group of members contributes disproportionately to overall check-in activity, indicating a core segment of highly engaged users.

#**Business Value:**  
#These members may be strong candidates for:
#- Loyalty recognition
#- VIP experiences
#- Targeted retention strategies

df['member_name'] = (
    df['first_name'].fillna('').astype(str).str.strip()
    + " " +
    df['last_name'].fillna('').astype(str).str.strip()
).str.strip()

dining_visits = (
    df[df['scan_location'] == 'Dining App']
    .groupby('member_name')
    .size()
    .reset_index(name='dining_visits')
    .sort_values(by='dining_visits', ascending=False)
)

print(dining_visits.head(10))

            member_name  dining_visits
3169     Sandy Edgerley             14
240   Andrew Richardson             13
2407        Matt Belkin             12
2441    Matthew Lucerto             12
3306       Sinem Atahan             12
1194      Frank Laukien             11
2451     Matthias Kiehm             11
3524   Tina Eliassi-Rad             11
2781          Noah Cate             10
2526  Michael Arnheiter             10


In [16]:
## Dining App vs General Check-Ins
#This analysis compares check-ins associated with dining reservations versus general member activity.

#Peoplevine Check-In = lounge usage/general check-ins, Dining App = Dining visits


df['member_name'] = (
    df['first_name'].fillna('').astype(str).str.strip()
    + " " +
    df['last_name'].fillna('').astype(str).str.strip()
).str.strip()

member_visit_breakdown = (
    df.groupby(['member_name', 'scan_location'])
    .size()
    .unstack(fill_value=0)
)

for col in ['Dining App', 'Peoplevine Check-In']:
    if col not in member_visit_breakdown.columns:
        member_visit_breakdown[col] = 0

member_visit_breakdown['total_visits'] = (
    member_visit_breakdown['Dining App'] + member_visit_breakdown['Peoplevine Check-In']
)

member_visit_breakdown['dining_ratio'] = (
    member_visit_breakdown['Dining App'] / member_visit_breakdown['total_visits']
)

member_visit_breakdown['member_type'] = member_visit_breakdown['dining_ratio'].apply(
    lambda x: 'Dining-Focused' if x >= 0.6
    else ('Balanced User' if x >= 0.4 else 'Lounge-Focused')
)

top_revenue_drivers = member_visit_breakdown.sort_values(by='Dining App', ascending=False)

print(top_revenue_drivers.head(10))

scan_location      65.249.150.194  Control Panel  Dining App  Hotel App  \
member_name                                                               
Sandy Edgerley                  0              0          14          0   
Andrew Richardson               0              0          13          0   
Matt Belkin                     0              0          12          0   
Matthew Lucerto                 0              0          12          0   
Sinem Atahan                    0              0          12          0   
Frank Laukien                   0              0          11          0   
Matthias Kiehm                  0              0          11          0   
Tina Eliassi-Rad                0              0          11          0   
Michael Arnheiter               0              0          10          0   
Noah Cate                       0              0          10          0   

scan_location      PeopleVine Check-In  Peoplevine Check-In  total_visits  \
member_name           

In [47]:
## Peak Activity by Hour

#This analysis identifies the busiest hours of the day based on member check-ins.
#Compare Dining App vs Peoplevine Check-In

df['scan_date'] = pd.to_datetime(df['scan_date'], errors='coerce')

#**Insight:**  
#Peak activity occurs around hour 18 (6 PM), which aligns with evening usage patterns such as dining and social activity.

In [18]:
df['hour'] = df['scan_date'].dt.hour

In [19]:
hourly_traffic = (
    df.groupby(['hour', 'scan_location'])
    .size()
    .unstack(fill_value=0)
)

In [20]:
print(hourly_traffic)

scan_location  65.249.150.194  Control Panel  Dining App  Hotel App  \
hour                                                                  
0                           0              0           0          0   
5                           0              0           0          0   
6                           0              0           0          0   
7                           0              0          42          0   
8                           0              1         126          0   
9                           0              7          76          0   
10                          0              0          90          0   
11                          0              8         240          0   
12                          0              1         675          1   
13                          0              7         352          1   
14                          0              3          89          1   
15                          0              0          18          2   
16    

In [21]:
print(hourly_traffic.sort_values(by='Dining App', ascending=False).head(5))
print(hourly_traffic.sort_values(by='Peoplevine Check-In', ascending=False).head(5))

scan_location  65.249.150.194  Control Panel  Dining App  Hotel App  \
hour                                                                  
19                          0              0        1681          0   
18                          0              2        1513          3   
20                          0              0         984          0   
12                          0              1         675          1   
17                          9              6         501          0   

scan_location  PeopleVine Check-In  Peoplevine Check-In  
hour                                                     
19                               9                 4052  
18                              11                 6583  
20                               0                 1647  
12                               0                 3037  
17                               5                 5290  
scan_location  65.249.150.194  Control Panel  Dining App  Hotel App  \
hour                     

In [46]:
#Average dining check in amounts by day vs lounge check ins

df['date'] = df['scan_date'].dt.date
df['day_of_week'] = df['scan_date'].dt.day_name()

In [23]:
daily_counts = (
    df.groupby(['date', 'scan_location'])
    .size()
    .unstack(fill_value=0)
)

In [24]:
daily_counts = daily_counts.reset_index()

daily_counts['day_of_week'] = pd.to_datetime(daily_counts['date']).dt.day_name()

In [25]:
avg_by_day = (
    daily_counts.groupby('day_of_week')[['Dining App', 'Peoplevine Check-In']]
    .mean()
)

In [26]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

avg_by_day = avg_by_day.reindex(day_order)

In [27]:
print(avg_by_day)

scan_location  Dining App  Peoplevine Check-In
day_of_week                                   
Monday          45.384615           327.307692
Tuesday         71.230769           442.769231
Wednesday       75.538462           417.307692
Thursday        94.000000           543.090909
Friday          98.000000           542.923077
Saturday        98.357143           553.500000
Sunday          33.538462           296.153846


In [29]:
#Busiest Month during the period and then the busiest days in that month

#This assist with seasonal and daily insight for operations in staffing and predictability

df['month'] = df['scan_date'].dt.month
df['month_name'] = df['scan_date'].dt.month_name()

In [30]:
monthly_counts = (
    df.groupby('month_name')
    .size()
)

month_order = [
    'January', 'February', 'March', 'April', 'May', 'June',
    'July', 'August', 'September', 'October', 'November', 'December'
]

monthly_counts = monthly_counts.reindex(month_order)

print(monthly_counts)
print("\nBusiest month:", monthly_counts.idxmax())

month_name
January      14569.0
February         NaN
March            NaN
April            NaN
May              NaN
June             NaN
July             NaN
August           NaN
September        NaN
October          NaN
November     17800.0
December     17011.0
dtype: float64

Busiest month: November


In [31]:
#what was the busiest day in november?

df['scan_date'] = pd.to_datetime(df['scan_date'], errors='coerce')

In [32]:
df['month'] = df['scan_date'].dt.month
df['date'] = df['scan_date'].dt.date

In [33]:
november_df = df[df['month'] == 11]

In [34]:
daily_counts_nov = november_df.groupby('date').size().sort_values(ascending=False)

print(daily_counts_nov)

date
2025-11-01    863
2025-11-07    848
2025-11-08    847
2025-11-15    822
2025-11-13    814
2025-11-14    810
2025-11-21    809
2025-11-20    789
2025-11-22    766
2025-11-06    752
2025-11-04    725
2025-11-29    661
2025-11-12    630
2025-11-18    628
2025-11-19    603
2025-11-11    602
2025-11-05    600
2025-11-25    532
2025-11-17    514
2025-11-10    478
2025-11-24    475
2025-11-28    464
2025-11-03    454
2025-11-09    414
2025-11-23    409
2025-11-26    396
2025-11-02    382
2025-11-16    381
2025-11-30    332
dtype: int64


In [35]:
#Identify least active members → potential churn risk

#**Insight:**  
#A significant portion of members shows low engagement, representing potential churn risk.

#**Business Value:**  
#This group presents an opportunity for:
#- Re-engagement campaigns
#- Incentive offers (e.g., complimentary drinks)
#- Personalized outreach

visit_counts = (
    df.groupby('member_name')
    .size()
    .reset_index(name='total_visits')
)

In [37]:
least_active = visit_counts.sort_values(by='total_visits', ascending=True)

print("Least active members:")
print(least_active.head(10))

Least active members:
                 member_name  total_visits
3946              Julie Ling             1
6959          Shaina Goodwin             1
5726           Moritz Flogel             1
3064             Jake Powers             1
5715          Morgan Conarro             1
1289  Charles Bradford Scott             1
5713          Morgan Abraham             1
5711           Mora Babineau             1
3072             James Bartz             1
3074           James Bildner             1


In [39]:
#Add recency (when was their last visit)

last_visit = (
    df.groupby('member_name')['scan_date']
    .max()
    .reset_index(name='last_visit_date')
)

In [40]:
member_activity = visit_counts.merge(last_visit, on='member_name')

In [41]:
churn_risk = member_activity.sort_values(
    by=['total_visits', 'last_visit_date'],
    ascending=[True, True]
)

print(churn_risk.head(10))

               member_name  total_visits     last_visit_date
5901     Nicole DuFauchard             1 2025-11-01 11:32:07
7487       Tiger Henderson             1 2025-11-01 11:40:54
6716      Sakeenah Chapman             1 2025-11-01 12:34:57
5073            Mark Sneed             1 2025-11-01 13:26:17
7146          Stephen Hart             1 2025-11-01 13:43:00
7346            Tammie Koh             1 2025-11-01 14:20:19
2593          Gabriel Quek             1 2025-11-01 14:20:22
5269  Maura McCarthy Haney             1 2025-11-01 14:53:44
7542       Tom Bergenholtz             1 2025-11-01 15:08:22
3709            John Spino             1 2025-11-01 15:14:33


In [ ]:
## Key Findings

#- Peak activity occurs during evening hours (~6 PM), aligning with dining and social usage
#- Member engagement varies across time periods, with identifiable high-traffic days and months
#- Dining-related check-ins represent a significant portion of activity, indicating strong overlap between social and food & beverage usage

## Business Implications

#- Staffing should be optimized around peak evening hours
#- Opportunities exist to increase engagement during lower-traffic periods
#- Dining incentives (such as complimentary offers) could increase participation and overall revenue